In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

CLIENT_ID = "CLIENT-ID"
CLIENT_SECRET = "CLIENT-SECRET"

FETCH_AUDIO_FEATURES = False
TRACKS_PER_GENRE = 200

OUTPUT_FILE = "spotify_dataset.csv"
CHECKPOINT_FILE = "spotify_checkpoint.json"

# A broad starting list, not every Spotify genre.
GENRES = [
    "pop", "rock", "hip-hop", "rap", "r&b", "soul",
    "jazz", "blues", "classical", "country", "folk",
    "bluegrass", "electronic", "dance", "house",
    "techno", "trance", "dubstep", "ambient", "disco",
    "funk", "metal", "punk", "alternative", "indie",
    "reggae", "reggaeton", "ska", "latin", "salsa",
    "bachata", "bossa nova", "samba", "k-pop", "j-pop",
    "afrobeat", "afrobeats", "gospel", "soundtrack", "lo-fi",
]

COLUMNS = [
    "album_id",
    "album_name",
    "album_type",
    "album_release_date",
    "album_total_tracks",
    "album_explicit",
    "year",
    "genre",
    "track_id",
    "track_number",
    "track_name",
    "track_popularity",
    "track_explicit",
    "track_release_date",
    "duration_ms",
    "popularity",
    "artists_names",
    "artists_ids",
    "artist_popularity",
    "artist_genres",
    "artist_followers",
    "principal_artist_name",
    "principal_artist_id",
]

AUDIO_COLUMNS = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence",
]

ALL_COLUMNS = COLUMNS + AUDIO_COLUMNS

In [ ]:
# here is where we need to develop the ds w/the parameters above
class SpotifyClient:
    def __init__(self, client_id, client_secret, artist_cache):
        self.client_id = client_id
        self.client_secret = client_secret
        self.session = requests.Session()
        self.token = None
        self.token_expires_at = 0
        self.artist_cache = artist_cache
        self.audio_enabled = FETCH_AUDIO_FEATURES

    def renew_token(self):
        response = self.session.post(
            "https://accounts.spotify.com/api/token",
            auth=(self.client_id, self.client_secret),
            data={"grant_type": "client_credentials"},
            timeout=30,
        )
        response.raise_for_status()

        data = response.json()
        self.token = data["access_token"]
        self.token_expires_at = (
            time.monotonic() + data["expires_in"] - 60
        )

    def get(self, path, params=None):
        auth_retried = False
        server_retries = 0

        while True:
            if (
                self.token is None
                or time.monotonic() >= self.token_expires_at
            ):
                self.renew_token()

            # Gentle pacing; rate limits can still occur.
            time.sleep(1)

            response = self.session.get(
                f"https://api.spotify.com/v1{path}",
                headers={
                    "Authorization": f"Bearer {self.token}"
                },
                params=params,
                timeout=30,
            )

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")

                try:
                    wait_seconds = max(1, float(retry_after))
                except (TypeError, ValueError):
                    wait_seconds = 60

                wait_seconds += 1
                print(
                    f"Rate limited. Waiting {wait_seconds:.0f} "
                    "seconds before retrying...",
                    flush=True,
                )
                time.sleep(wait_seconds)
                continue

            if response.status_code == 401 and not auth_retried:
                self.renew_token()
                auth_retried = True
                continue

            if response.status_code in (500, 502, 503, 504):
                if server_retries < 4:
                    time.sleep(2 ** (server_retries + 1))
                    server_retries += 1
                    continue

            response.raise_for_status()
            return response.json()

    def get_artist(self, artist_id):
        # Cache persists across genres and notebook runs.
        if artist_id not in self.artist_cache:
            self.artist_cache[artist_id] = self.get(
                f"/artists/{artist_id}"
            )

        return self.artist_cache[artist_id]

    def build_row(self, track):
        album = track.get("album") or {}
        artists = track.get("artists") or []
        principal = artists[0] if artists else {}
        release_date = album.get("release_date")

        details_by_id = {
            artist["id"]: self.get_artist(artist["id"])
            for artist in artists
            if artist.get("id")
        }

        principal_details = details_by_id.get(
            principal.get("id"), {}
        )

        genre_lists = [
            details["genres"]
            for details in details_by_id.values()
            if details.get("genres") is not None
        ]

        combined_genres = (
            sorted({
                genre
                for genres in genre_lists
                for genre in genres
            })
            if genre_lists
            else None
        )

        audio = {}

        if self.audio_enabled:
            try:
                audio = self.get(
                    f"/audio-features/{track['id']}"
                ) or {}
            except requests.HTTPError as error:
                if error.response.status_code == 403:
                    print(
                        "Audio Features access denied; "
                        "leaving audio columns blank."
                    )
                    self.audio_enabled = False
                else:
                    raise

        row = {
            "album_id": album.get("id"),
            "album_name": album.get("name"),
            "album_type": album.get("album_type"),
            "album_release_date": release_date,
            "album_total_tracks": album.get("total_tracks"),

            # Requires checking every album track; unknown here.
            "album_explicit": None,

            "year": int(release_date[:4]) if release_date else None,

            # Combined artist genres, not track-level labels.
            "genre": combined_genres,

            "track_id": track.get("id"),
            "track_number": track.get("track_number"),
            "track_name": track.get("name"),
            "track_popularity": track.get("popularity"),
            "track_explicit": track.get("explicit"),

            # Album release date, not necessarily original release.
            "track_release_date": release_date,

            "duration_ms": track.get("duration_ms"),
            "popularity": track.get("popularity"),
            "artists_names": [a.get("name") for a in artists],
            "artists_ids": [a.get("id") for a in artists],

            # These describe the first credited artist.
            "artist_popularity": principal_details.get("popularity"),
            "artist_genres": principal_details.get("genres"),
            "artist_followers": (
                principal_details.get("followers") or {}
            ).get("total"),

            "principal_artist_name": principal.get("name"),
            "principal_artist_id": principal.get("id"),
        }

        row.update({
            column: audio.get(column)
            for column in AUDIO_COLUMNS
        })

        return row


def load_progress():
    path = Path(CHECKPOINT_FILE)

    if path.exists():
        with path.open("r", encoding="utf-8") as file:
            return json.load(file)

    return {
        "tracks": {},
        "artists": {},
        "offsets": {},
        "completed_queries": [],
    }


def save_progress(state):
    # Replace the checkpoint only after writing it successfully.
    checkpoint = Path(CHECKPOINT_FILE)
    temporary = checkpoint.with_suffix(".tmp")

    with temporary.open("w", encoding="utf-8") as file:
        json.dump(state, file, ensure_ascii=False)

    temporary.replace(checkpoint)

    df = pd.DataFrame(
        list(state["tracks"].values()),
        columns=ALL_COLUMNS,
    )

    for column in [
        "genre", "artists_names", "artists_ids", "artist_genres"
    ]:
        df[column] = df[column].map(
            lambda value: (
                json.dumps(value, ensure_ascii=False)
                if isinstance(value, list)
                else value
            )
        )

    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")


def main():
    if (
        not CLIENT_ID
        or not CLIENT_SECRET
        or CLIENT_ID == "YOUR_CLIENT_ID"
        or CLIENT_SECRET == "YOUR_CLIENT_SECRET"
    ):
        raise ValueError("Enter your Spotify credentials first.")

    if not 1 <= TRACKS_PER_GENRE <= 1000:
        raise ValueError("TRACKS_PER_GENRE must be 1–1000.")

    state = load_progress()
    client = SpotifyClient(
        CLIENT_ID, CLIENT_SECRET, state["artists"]
    )

    try:
        for genre in GENRES:
            query = f'genre:"{genre}"'

            # Include collection settings in the progress key.
            progress_key = f"{query}|US|{TRACKS_PER_GENRE}"

            if progress_key in state["completed_queries"]:
                print(f"Already completed: {genre}")
                continue

            offset = state["offsets"].get(progress_key, 0)
            print(f"Collecting: {genre}, starting at {offset}")

            while offset < TRACKS_PER_GENRE:
                page_size = min(10, TRACKS_PER_GENRE - offset)

                result = client.get(
                    "/search",
                    params={
                        "q": query,
                        "type": "track",
                        "market": "US",
                        "limit": page_size,
                        "offset": offset,
                    },
                )

                page = result["tracks"]
                tracks = page.get("items", [])

                if not tracks:
                    break

                for track in tracks:
                    if not track or not track.get("id"):
                        continue

                    track_id = track["id"]

                    if track_id not in state["tracks"]:
                        state["tracks"][track_id] = (
                            client.build_row(track)
                        )

                offset += len(tracks)
                state["offsets"][progress_key] = offset
                save_progress(state)

                print(
                    f"  {genre}: {offset} search results processed; "
                    f"{len(state['tracks'])} unique tracks saved"
                )

                if not page.get("next"):
                    break

            state["completed_queries"].append(progress_key)
            save_progress(state)

    finally:
        # Also save if interrupted or a request fails.
        save_progress(state)
        client.session.close()
        print(
            f"\nSaved {len(state['tracks'])} unique tracks "
            f"to {OUTPUT_FILE}"
        )


if __name__ == "__main__":
    main()
